# Ultramarathon Performance Visualizations — Part A
**Group 17 — Luiz Samelo, Maximilian Staudacher, Rayudu Muralikrishna**  
*Information Visualization (VU 2.0), TU Wien, 2026*

- **Viz 1**: Choropleth map — avg speed by country with year slider
- **Viz 2**: Top 10 national dominance line chart

## 0. Imports & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import country_converter as coco
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/TWO_CENTURIES_OF_UM_RACES.csv')
print('Shape:', df.shape)
df.head()

## 1. Data Cleaning

In [ ]:
df.columns = [
    'year', 'event_dates', 'race_name', 'race_length', 'num_finishers',
    'athlete_performance', 'athlete_club', 'athlete_country',
    'athlete_year_of_birth', 'athlete_gender', 'athlete_age_category',
    'athlete_avg_speed', 'athlete_id'
]
df['year'] = pd.to_numeric(df['year'], errors='coerce')
df['athlete_avg_speed'] = pd.to_numeric(df['athlete_avg_speed'], errors='coerce')
df['athlete_year_of_birth'] = pd.to_numeric(df['athlete_year_of_birth'], errors='coerce')

cc = coco.CountryConverter()

def convert_country(series, to='name_short', src='IOC'):
    return series.apply(lambda x: cc.convert(x, src=src, to=to) if pd.notna(x) else None)

print('Columns OK:', df.columns.tolist())

## Viz 1 — Choropleth Map: Avg Speed by Country with Year Slider

In [ ]:
df_clean = df[
    (df['race_length'] == '50km') &
    (df['year'] >= 1970) & (df['year'] <= 2022) &
    (df['athlete_avg_speed'] > 0) &
    (df['athlete_country'].notna())
].copy()

df_clean['iso3'] = convert_country(df_clean['athlete_country'], to='ISO3')
df_clean['country_name'] = convert_country(df_clean['athlete_country'], to='name_short')
df_clean = df_clean[df_clean['iso3'] != 'not found']

choropleth_data = (
    df_clean.groupby(['year', 'iso3', 'country_name'])
    .agg(avg_speed=('athlete_avg_speed', 'mean'), finishers=('athlete_avg_speed', 'count'))
    .reset_index()
)
choropleth_data = choropleth_data[choropleth_data['finishers'] >= 10]
choropleth_data['avg_speed'] = choropleth_data['avg_speed'].round(2)
print('Choropleth data shape:', choropleth_data.shape)

In [ ]:
fig1 = px.choropleth(
    choropleth_data,
    locations='iso3',
    locationmode='ISO-3',
    color='avg_speed',
    animation_frame='year',
    color_continuous_scale='RdYlGn',
    range_color=[6, 14],
    hover_name='country_name',
    hover_data={'avg_speed': ':.2f', 'finishers': True, 'iso3': False, 'year': False},
    labels={'avg_speed': 'Avg Speed (km/h)', 'finishers': 'Finishers'},
    title='Average 50km Ultra-Marathon Speed by Country (1970-2022)'
)
fig1.update_layout(
    geo=dict(showframe=False, showcoastlines=True, projection_type='natural earth'),
    coloraxis_colorbar=dict(title='Avg Speed<br>(km/h)'),
    title_font_size=18, height=550,
    margin=dict(l=0, r=0, t=60, b=0)
)
fig1.show()
fig1.write_html('../Viz/viz1_choropleth_speed_by_country.html')
print('Saved: Viz/viz1_choropleth_speed_by_country.html')

## Viz 2 — Top 10 National Dominance Line Chart

In [ ]:
df_all = df[
    (df['year'] >= 1970) & (df['year'] <= 2022) &
    df['athlete_country'].notna()
].copy()

df_all['iso3'] = convert_country(df_all['athlete_country'], to='ISO3')
df_all['country_name'] = convert_country(df_all['athlete_country'], to='name_short')
df_all = df_all[df_all['iso3'] != 'not found']

dominance = df_all.groupby(['year', 'country_name']).size().reset_index(name='finishers')
top10 = dominance.groupby('country_name')['finishers'].sum().nlargest(10).index.tolist()
print('Top 10 countries:', top10)

dominance_top10 = dominance[dominance['country_name'].isin(top10)].copy()
dominance_top10 = dominance_top10.sort_values(['country_name', 'year'])
dominance_top10['finishers_smooth'] = (
    dominance_top10.groupby('country_name')['finishers']
    .transform(lambda x: x.rolling(3, min_periods=1).mean())
)
print('Dominance data shape:', dominance_top10.shape)

In [ ]:
fig2 = px.line(
    dominance_top10,
    x='year', y='finishers_smooth',
    color='country_name',
    hover_data={'finishers': True, 'finishers_smooth': ':.0f'},
    labels={
        'year': 'Year',
        'finishers_smooth': 'Finishers (3-yr avg)',
        'country_name': 'Country',
        'finishers': 'Raw finishers'
    },
    title='Top 10 Countries by Ultra-Marathon Finishers Over Time (1970-2022)',
    color_discrete_sequence=px.colors.qualitative.Bold
)
for decade in [1980, 1990, 2000, 2010, 2020]:
    fig2.add_vline(x=decade, line_dash='dot', line_color='gray', opacity=0.5)
    fig2.add_annotation(x=decade, y=1, yref='paper', text=str(decade),
                        showarrow=False, font=dict(size=10, color='gray'), yanchor='top')
fig2.update_layout(
    hovermode='x unified',
    legend=dict(title='Country', orientation='v', x=1.01, y=1),
    xaxis=dict(title='Year', dtick=5),
    yaxis=dict(title='Number of Finishers (3-yr rolling avg)'),
    title_font_size=18, height=550,
    plot_bgcolor='white', paper_bgcolor='white'
)
fig2.update_xaxes(showgrid=True, gridcolor='#f0f0f0')
fig2.update_yaxes(showgrid=True, gridcolor='#f0f0f0')
fig2.show()
fig2.write_html('../Viz/viz2_national_dominance.html')
print('Saved: Viz/viz2_national_dominance.html')